
# Enrich Dosen IDs + Map Scival Authors → Dosen (ID-first, Name-fallback)
This notebook performs two pipelines:

**A. Enrichment**  
`dosen_data.csv` ⟵ merge from `data_dosen.csv` (adds **Scopus ID**, **Sinta ID**, **Scholar ID**) using normalized name mapping.

**B. Author Mapping (Scival)**  
Maps Scival `authors` + `scopus author ids` to enriched dosen. Strategy:
1) **Direct ID match** (Scopus Author ID) → exact & trusted  
2) **Tiered name match**: exact → rule-based (initials+lastname, token-set) → fuzzy (RapidFuzz)

**Outputs**
- `dosen_enriched.csv` – dosen internal + IDs (scopus/sinta/scholar)
- `author_matches.csv`, `author_ambiguous.csv`, `author_unmatched.csv` – mapping reports


## 1) Paths

In [36]:
from pathlib import Path
BASE = Path().cwd() # sesuaikan jika perlu
DOSEN_CSV = BASE / "file_tabulars/dosen_data.csv"
DATA_DOSEN_CSV = BASE / "file_tabulars/data_dosen.csv"
SCIVAL = BASE / "file_tabulars/scival_scraped.csv"

# Outputs
DOSEN_ENRICHED = BASE / "file_tabulars/dosen_data.csv"
MATCHES_CSV    = BASE / "file_tabulars/author_matches.csv"
AMBIG_CSV      = BASE / "file_tabulars/author_ambiguous.csv"
UNMATCH_CSV    = BASE / "file_tabulars/author_unmatched.csv"

DOSEN_CSV, DATA_DOSEN_CSV, SCIVAL


(WindowsPath('C:/Users/rizky/OneDrive/Dokumen/GitHub/Tugas_Akhir/scraping/file_tabulars/dosen_data.csv'),
 WindowsPath('C:/Users/rizky/OneDrive/Dokumen/GitHub/Tugas_Akhir/scraping/file_tabulars/data_dosen.csv'),
 WindowsPath('C:/Users/rizky/OneDrive/Dokumen/GitHub/Tugas_Akhir/scraping/file_tabulars/scival_scraped.csv'))

## 2) Imports & Helpers

In [16]:
import pandas as pd
import re
import unicodedata
from typing import List
from collections import Counter

try:
    from rapidfuzz import fuzz, process as rf_process
    HAVE_RF = True
except Exception:
    HAVE_RF = False

def strip_accents(s: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFD", s) if unicodedata.category(ch) != "Mn")

TITLE_PREFIX = re.compile(r"^\s*(prof\.?|drs\.?|dra\.?|dr\.?|ir\.?|h\.?|hj\.?)\s+", re.IGNORECASE)

def normalize_name_generic(raw: str) -> str:
    if pd.isna(raw):
        return ""
    s = strip_accents(str(raw)).strip()
    # remove degrees after comma
    s = s.split(",")[0]
    # remove titles repeatedly
    prev = None
    while prev != s:
        prev = s
        s = TITLE_PREFIX.sub("", s).strip()
    # remove dots, compress spaces, lower
    s = s.replace(".", " ")
    s = re.sub(r"\s+", " ", s)
    return s.lower().strip()

def normalize_scival_author(raw: str) -> str:
    "Turn 'Lastname, F.M.' → 'f m lastname' (lower, no dots, no titles)"
    if pd.isna(raw):
        return ""
    s = strip_accents(str(raw)).strip()
    parts = [p.strip() for p in s.split(",")]
    if len(parts) >= 2:
        last, first = parts[0], parts[1]
        s = f"{first} {last}"
    # remove academic prefixes if any (rare in Scopus 'authors', but safe)
    prev = None
    while prev != s:
        prev = s
        s = TITLE_PREFIX.sub("", s).strip()
    s = s.replace(".", " ")
    s = re.sub(r"\s+", " ", s)
    return s.lower().strip()

def pick_col(df, candidates, required=False):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Couldn't find any of these columns: {candidates}")
    return None

def rule_based_aliases(norm_name: str) -> List[str]:
    toks = [t for t in re.split(r"\s+", norm_name) if t]
    if not toks: 
        return [norm_name]
    initials = " ".join(t[0] for t in toks if t)
    last = toks[-1]
    variants = {norm_name, f"{initials} {last}".strip(), f"{last} {initials}".strip()}
    variants.add(" ".join(sorted(set(toks))))
    return list(variants)


## 3) Load data

In [17]:

dosen_internal = pd.read_csv(DOSEN_CSV, encoding="utf-8-sig")
data_dosen     = pd.read_csv(DATA_DOSEN_CSV, encoding="utf-8-sig")
scival         = pd.read_csv(SCIVAL, encoding="utf-8-sig")

print("dosen_internal cols:", list(dosen_internal.columns))
print("data_dosen cols:", list(data_dosen.columns))
print("scival cols:", list(scival.columns)[:20], "...")


dosen_internal cols: ['id_sdm', 'nama_dosen', 'nidn', 'jabatan_akademik', 'pendidikan_tertinggi', 'status_ikatan_kerja', 'status_aktivitas', 'nama_pt', 'nama_prodi', 'jumlah_penelitian', 'jumlah_pengabdian', 'jumlah_karya_ilmiah', 'jumlah_paten', '_norm_name', 'nip', 'sample_full_name']
data_dosen cols: ['Nama', 'NIDN', 'NIP', 'Detail_URL', 'Photo_URL', 'QR_Code_URL', 'ID_Sinta', 'ID_Scopus', 'ID_Scholar', 'Program_Studi']
scival cols: ['title', 'authors', 'number of authors', 'scopus author ids', 'year', 'full date', 'scopus source title', 'volume', 'issue', 'pages', 'article number', 'issn', 'source id', 'source type', 'language', 'snip (publication year)', 'snip percentile (publication year) *', 'citescore (publication year)', 'citescore percentile (publication year) *', 'sjr (publication year)'] ...


## 4) Build ID mapping from `data_dosen.csv` → normalized name

In [22]:

# Try to detect likely ID columns in data_dosen (case-insensitive variants)
col_name = pick_col(data_dosen, ["nama_dosen","nama","Nama Dosen","Nama"], required=True)
col_scopus = pick_col(data_dosen, ["ID_Scopus","id_scopus","ScopusID","Scopus_Id","scopus_id"])
col_sinta  = pick_col(data_dosen, ["ID_Sinta","id_sinta","SintaID","sinta_id"])
col_scholar= pick_col(data_dosen, ["ID_Scholar","id_scholar","ScholarID","scholar_id","ID_Scholar_Google","google_scholar_id"])

data_dosen["_norm_name"] = data_dosen[col_name].map(normalize_name_generic)

agg_dict = {}
if col_scopus: agg_dict["scopus_id"] = (col_scopus, lambda s: next((x for x in s if pd.notna(x) and str(x).strip()!=''), pd.NA))
if col_sinta:  agg_dict["sinta_id"]  = (col_sinta,  lambda s: next((x for x in s if pd.notna(x) and str(x).strip()!=''), pd.NA))
if col_scholar:agg_dict["scholar_id"]= (col_scholar,lambda s: next((x for x in s if pd.notna(x) and str(x).strip()!=''), pd.NA))

id_map = data_dosen.groupby("_norm_name", dropna=False).agg(**agg_dict).reset_index() if agg_dict else pd.DataFrame(columns=["_norm_name"])
id_map.head(10)


,_norm_name,scopus_id,sinta_id,scholar_id
0,(h c ) h abdul halim iskandar,<NA>,<NA>,<NA>
1,a burhanuddin kusuma nugraha,<NA>,6857545,<NA>
2,a grummy wailanduw,56013141800,74808,I81f8eQAAAAJ
3,a'rasy fahrullah,https://www.scopus.com/authid/detail.uri?autho...,http://sinta.ristekbrin.go.id/authors/detail?i...,https://scholar.google.co.id/citations?user=lH...
4,a'yunin sofro,57200365842,6010310,<NA>
5,abadi,7409825061,5996798,QIE6Mf4AAAAJ
6,abd kholiq,https://www.scopus.com/authid/detail.uri?autho...,5994616,https://scholar.google.com/citations?hl=en&use...
7,abdiyah amudi,<NA>,6164108,5DByoLMAAAAJ
8,abdul aziz hakim,57247713000,6716469,https://scholar.google.co.id/citations?hl=id&u...
9,abdul aziz khoiri,<NA>,6858765,76lZ4FsAAAAJ


## 5) Enrich `dosen_data.csv` with IDs (by normalized name)

In [24]:

# Normalize internal dosen names
name_col_internal = pick_col(dosen_internal, ["nama_dosen","nama","Nama Dosen","Nama"], required=True)
dosen_internal["_norm_name"] = dosen_internal[name_col_internal].map(normalize_name_generic)

enriched = dosen_internal.merge(id_map, how="left", left_on="_norm_name", right_on="_norm_name")

# Keep columns order: original + new ID columns at the end
ordered = list(dosen_internal.columns) + [c for c in ["scopus_id","sinta_id","scholar_id"] if c in enriched.columns and c not in dosen_internal.columns]
enriched = enriched[ordered]

enriched.to_csv(DOSEN_ENRICHED, index=False, encoding="utf-8-sig")
print("Saved:", DOSEN_ENRICHED)
enriched.head(10)


Saved: C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\dosen_data.csv


,id_sdm,nama_dosen,nidn,jabatan_akademik,pendidikan_tertinggi,status_ikatan_kerja,status_aktivitas,nama_pt,nama_prodi,jumlah_penelitian,jumlah_pengabdian,jumlah_karya_ilmiah,jumlah_paten,_norm_name,nip,sample_full_name,scopus_id,sinta_id,scholar_id
0,sLZzYTPWD4CeqSEAVN-2cT_TIGHN2WrXWbgM3KWj3vUrDC...,Nurul Laili,NaN,NaN,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,0,0,0,0,nurul laili,2.025092e+08,"Nurul Laili, S.Pd., M.Sc.",<NA>,<NA>,<NA>
1,RmxdOmO31i_M8sDBvKw_GMni0V6WrrsplULBC16QoFHhsp...,Sabrina Amelialevi,NaN,NaN,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,0,0,0,0,sabrina amelialevi,2.025092e+08,"Sabrina Amelialevi, S.Kom., M.Kom.",<NA>,<NA>,<NA>
2,e4u0fxTGIXZk6OVg1D7m069hCuRXliun1mKiHaKuuctsYH...,Atik Wintarti,12106608.0,Lektor Kepala,S3,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,11,6,74,12,atik wintarti,1.966101e+17,"Dr. Atik Wintarti, M.Kom.",57053275600,5995948,lFCI_vsAAAAJ
3,ug18IBRkRo2_1vVjCnaL2pNGZVIi9vtromIBpKjtE9X1x2...,Siska Puspitaningsih,725129501.0,NaN,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,0,4,11,1,siska puspitaningsih,2.025092e+08,"Siska Puspitaningsih, S.Kom., M.Kom.",<NA>,<NA>,<NA>
4,XSZZdZfK-0lj0Q0D8ItXeI3YO8eS_s4kerNAxQfMPBwYdG...,Heri Purnawan,706069301.0,NaN,S3,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,0,4,12,1,heri purnawan,2.025092e+08,"Dr. Heri Purnawan, S.Si., M.Si.",<NA>,<NA>,<NA>
5,FYVyQepNiSlB6DaekFokRICIT-HzvzlvimBRQxVrrnEmFJ...,Belgis Ainatul Iza,NaN,NaN,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,0,0,0,0,belgis ainatul iza,2.025092e+08,"Belgis Ainatul Iza, S.Si, M.Mat.",<NA>,<NA>,<NA>
6,51rEX5Z49Gjhoqhh3pOnCQ5S2QvV3JYEACF0lOljO1g5qu...,Ibnu Febry Kurniawan,18028801.0,Asisten Ahli,S3,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,12,12,26,2,ibnu febry kurniawan,1.988022e+17,"Ibnu Febry Kurniawan, S.Kom., M.Sc., Ph.D.",57201757812,6008890,BIrtbD8AAAAJ
7,UvM-UyzM2-1YAa5s4zKACptXeqMUvR5JVzt4Ioo2VFG3pi...,Dinda Galuh Guminta,11129602.0,NaN,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,2,0,0,1,dinda galuh guminta,1.996121e+17,"Dinda Galuh Guminta, M.Stat.",<NA>,6903529,<NA>
8,PGt25k6rqQshJfmGoStYk8yF1W2TgWV3Gfg8tRkOblCdSm...,Ulfa Siti Nuraini,4109601.0,NaN,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,4,3,4,0,ulfa siti nuraini,2.023101e+08,"Ulfa Siti Nuraini, S.Stat., M.Stat.",57221534129,6869762,q-sbahIAAAAJ
9,5HLmrZPYpQ7E_bYFJPAMzgtI2hNLPhER5JBmHJ8dB53rVo...,Yuliani Puji Astuti,31077804.0,Lektor,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,12,9,32,9,yuliani puji astuti,1.978073e+17,"Yuliani Puji Astuti, S.Si., M.Si.",57188990409,6010301,Tz8EfXcAAAAJ


## 6) Prepare Scival authors (explode & normalize)

In [25]:

# Detect author columns
authors_col = pick_col(scival, ["authors","Authors"], required=True)
author_ids_col = pick_col(scival, ["scopus author ids","Scopus Author Ids","Author_IDs","author_ids"])

def split_pipe(s: str) -> List[str]:
    if pd.isna(s) or not str(s).strip():
        return []
    return [p.strip() for p in str(s).split("|") if str(p).strip()]

scival["__authors"] = scival[authors_col].map(split_pipe)
scival["__author_ids"] = scival[author_ids_col].map(split_pipe) if author_ids_col else [[] for _ in range(len(scival))]

exp = scival[[authors_col,"__authors","__author_ids"]].explode(["__authors","__author_ids"], ignore_index=False).reset_index().rename(columns={"index":"pub_index","__authors":"author_name_raw","__author_ids":"author_id_raw"})
exp["author_name_norm"] = exp["author_name_raw"].map(normalize_scival_author)
exp.head(10)


,pub_index,authors,author_name_raw,author_id_raw,author_name_norm
0,0,"Wijayati, D.T.| Rahman, Z.| Fahrullah, A.| Rah...","Wijayati, D.T.",57202647254,d t wijayati
1,0,"Wijayati, D.T.| Rahman, Z.| Fahrullah, A.| Rah...","Rahman, Z.",57202893305,z rahman
2,0,"Wijayati, D.T.| Rahman, Z.| Fahrullah, A.| Rah...","Fahrullah, A.",57203846106,a fahrullah
3,0,"Wijayati, D.T.| Rahman, Z.| Fahrullah, A.| Rah...","Rahman, M.F.W.",58949238400,m f w rahman
4,0,"Wijayati, D.T.| Rahman, Z.| Fahrullah, A.| Rah...","Arifah, I.D.C.",57395409700,i d c arifah
5,0,"Wijayati, D.T.| Rahman, Z.| Fahrullah, A.| Rah...","Kautsar, A.",57202283774,a kautsar
6,1,"Tjahjadi, B.| Soewarno, N.| Hariyati, H.| Nafi...","Tjahjadi, B.",57191278873,b tjahjadi
7,1,"Tjahjadi, B.| Soewarno, N.| Hariyati, H.| Nafi...","Soewarno, N.",57203927339,n soewarno
8,1,"Tjahjadi, B.| Soewarno, N.| Hariyati, H.| Nafi...","Hariyati, H.",57211274374,hariyati
9,1,"Tjahjadi, B.| Soewarno, N.| Hariyati, H.| Nafi...","Nafidah, L.N.",57218212743,l n nafidah


## 7) Build dosen lookup (name & Scopus ID)

In [26]:

# For ID match: scopus_id should be comparable to Scopus Author ID (both as strings)
enriched["scopus_id_str"] = enriched.get("scopus_id", pd.Series([pd.NA]*len(enriched))).astype(str).str.strip()
dosen_lookup = enriched[["_norm_name", name_col_internal, "scopus_id_str"]].drop_duplicates()

print(dosen_lookup.head(10))


             _norm_name            nama_dosen scopus_id_str
0           nurul laili           Nurul Laili          <NA>
1    sabrina amelialevi    Sabrina Amelialevi          <NA>
2         atik wintarti         Atik Wintarti   57053275600
3  siska puspitaningsih  Siska Puspitaningsih          <NA>
4         heri purnawan         Heri Purnawan          <NA>
5    belgis ainatul iza    Belgis Ainatul Iza          <NA>
6  ibnu febry kurniawan  Ibnu Febry Kurniawan   57201757812
7   dinda galuh guminta   Dinda Galuh Guminta          <NA>
8     ulfa siti nuraini     Ulfa Siti Nuraini   57221534129
9   yuliani puji astuti   Yuliani Puji Astuti   57188990409


## 8) Matching logic: ID-first → name rules → fuzzy

In [27]:

def rule_variants(s: str) -> List[str]:
    return rule_based_aliases(s)

def match_row(author_norm: str, author_id: str, dosen_df: pd.DataFrame, fuzzy_threshold=90):
    # 1) Direct Scopus ID match if present
    if author_id and str(author_id).strip().lower() != "nan":
        id_matches = dosen_df[dosen_df["scopus_id_str"].astype(str).str.strip() == str(author_id).strip()]
        if len(id_matches) == 1:
            r = id_matches.iloc[0]
            return {"strategy":"id", "matched_name": r[name_col_internal], "matched_norm": r["_norm_name"], "score": 100}
        elif len(id_matches) > 1:
            return {"strategy":"id-ambiguous", "candidates": id_matches[name_col_internal].tolist(), "score": 100}
    # 2) Exact norm name
    exact = dosen_df[dosen_df["_norm_name"] == author_norm]
    if len(exact) == 1:
        r = exact.iloc[0]
        return {"strategy":"exact", "matched_name": r[name_col_internal], "matched_norm": r["_norm_name"], "score": 99}
    elif len(exact) > 1:
        return {"strategy":"exact-ambiguous", "candidates": exact[name_col_internal].tolist(), "score": 99}
    # 3) Rule-based variants
    vars_ = rule_variants(author_norm)
    rb = dosen_df[dosen_df["_norm_name"].isin(vars_)]
    if len(rb) == 1:
        r = rb.iloc[0]
        return {"strategy":"rule", "matched_name": r[name_col_internal], "matched_norm": r["_norm_name"], "score": 95}
    elif len(rb) > 1:
        return {"strategy":"rule-ambiguous", "candidates": rb[name_col_internal].tolist(), "score": 95}
    # 4) Fuzzy
    if HAVE_RF:
        choices = dosen_df["_norm_name"].tolist()
        best = rf_process.extractOne(author_norm, choices, scorer=fuzz.token_set_ratio)
        if best is not None:
            best_norm, best_score, _ = best
            if best_score >= fuzzy_threshold:
                r = dosen_df[dosen_df["_norm_name"] == best_norm].iloc[0]
                return {"strategy":"fuzzy", "matched_name": r[name_col_internal], "matched_norm": r["_norm_name"], "score": int(best_score)}
    return {"strategy":"unmatched"}

rows = []
for i, r in exp.iterrows():
    res = match_row(r["author_name_norm"], str(r.get("author_id_raw","")).strip(), dosen_lookup, fuzzy_threshold=90)
    rows.append({
        "pub_index": r["pub_index"],
        "author_name_raw": r["author_name_raw"],
        "author_name_norm": r["author_name_norm"],
        "author_id_raw": r.get("author_id_raw", None),
        "match_strategy": res.get("strategy"),
        "matched_dosen_name": res.get("matched_name"),
        "matched_norm_name": res.get("matched_norm"),
        "score": res.get("score"),
        "candidates": "|".join(res.get("candidates", [])) if "candidates" in res else None
    })

matches = pd.DataFrame(rows)
matches.head(20)


,pub_index,author_name_raw,author_name_norm,author_id_raw,match_strategy,matched_dosen_name,matched_norm_name,score,candidates
0,0,"Wijayati, D.T.",d t wijayati,57202647254,unmatched,None,None,NaN,None
1,0,"Rahman, Z.",z rahman,57202893305,unmatched,None,None,NaN,None
2,0,"Fahrullah, A.",a fahrullah,57203846106,unmatched,None,None,NaN,None
3,0,"Rahman, M.F.W.",m f w rahman,58949238400,unmatched,None,None,NaN,None
4,0,"Arifah, I.D.C.",i d c arifah,57395409700,unmatched,None,None,NaN,None
5,0,"Kautsar, A.",a kautsar,57202283774,unmatched,None,None,NaN,None
6,1,"Tjahjadi, B.",b tjahjadi,57191278873,unmatched,None,None,NaN,None
7,1,"Soewarno, N.",n soewarno,57203927339,unmatched,None,None,NaN,None
8,1,"Hariyati, H.",hariyati,57211274374,unmatched,None,None,NaN,None
9,1,"Nafidah, L.N.",l n nafidah,57218212743,unmatched,None,None,NaN,None


## 9) Summary & Save

In [28]:

matched_ok = matches[matches["match_strategy"].isin(["id","exact","rule","fuzzy"])]
ambig = matches[matches["match_strategy"].str.contains("ambiguous", na=False)]
unmatched = matches[matches["match_strategy"]=="unmatched"]

summary = {
    "total_author_rows": len(matches),
    "matched": len(matched_ok),
    "ambiguous": len(ambig),
    "unmatched": len(unmatched),
    "match_rate_%": round(100.0*len(matched_ok)/max(1,len(matches)), 2)
}
print("Summary:", summary)

matches.to_csv(MATCHES_CSV, index=False, encoding="utf-8-sig")
ambig.to_csv(AMBIG_CSV, index=False, encoding="utf-8-sig")
unmatched.to_csv(UNMATCH_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print("-", DOSEN_ENRICHED)
print("-", MATCHES_CSV)
print("-", AMBIG_CSV)
print("-", UNMATCH_CSV)


Summary: {'total_author_rows': 12210, 'matched': 314, 'ambiguous': 0, 'unmatched': 11896, 'match_rate_%': 2.57}
Saved:
- C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\dosen_data.csv
- C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\author_matches.csv
- C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\author_ambiguous.csv
- C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\author_unmatched.csv


In [30]:
import pandas as pd
import re

# Load file
scival_path = BASE / "file_tabulars/scival_scraped.csv"
df = pd.read_csv(scival_path, encoding="utf-8-sig")

print("Before filter:", len(df))

# --- 1) Filter hanya yang ada 'Computer Science' di kolom THE field name ---
the_field_col = [c for c in df.columns if "times higher education" in c.lower() and "field name" in c.lower()][0]

# Gunakan regex agar bisa menangkap kasus seperti 'Computer Science| Education Studies'
pattern = re.compile(r"(^|\|)\s*Computer Science\s*(\||$)", re.IGNORECASE)
df = df[df[the_field_col].astype(str).apply(lambda x: bool(pattern.search(x)))]

print("After filter:", len(df))

# --- 2) Pilih hanya kolom relevan ---
keep_cols = [
    "title",
    "authors",
    "scopus author ids",
    "year",
    "scopus source title",
    "language",
    "citescore (publication year)",
    "citescore percentile (publication year) *",
    "sjr (publication year)",
    "times higher education (the) field name",
    "abstract_x",
    "abstract_y",
    "doi",
    "publication type",
    "open access",
    "country/region",
    "institutions",
]
keep_cols = [c for c in keep_cols if c in df.columns]
filtered_df = df[keep_cols].copy()

# --- 3) Simpan hasil bersih ---
out_path =  BASE / "file_tabulars/scival_filtered_compsci.csv"
filtered_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Saved filtered file:", out_path)
filtered_df.head()


Before filter: 2536
After filter: 453
Saved filtered file: C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\scival_filtered_compsci.csv


,title,authors,scopus author ids,year,scopus source title,language,citescore (publication year),citescore percentile (publication year) *,sjr (publication year),times higher education (the) field name,abstract_x,abstract_y,doi,publication type,open access,country/region,institutions
14,Covid Symptom Severity Using Decision Tree,"Rochmawati, N.| Hidayati, H.B.| Yamasari, Y.| ...",57195258934| 57194159504| 57193733535| 5720056...,2020.0,Proceeding - 2020 3rd International Conference...,English,-,-,-,Computer Science| Education Studies| Engineering,https://www.scopus.com/record/display.url?eid=...,Corona is a very contagious virus. In a pandem...,10.1109/ICVEE50212.2020.9243246,Conference Paper,Bronze,Indonesia,Universitas Airlangga| Universitas Negeri Sura...
21,Autonomous Robotic in Agriculture: A Review,"Rahmadian, R.| Widyartono, M.",57201853815| 57201863232,2020.0,Proceeding - 2020 3rd International Conference...,English,-,-,-,Engineering| Education Studies| Computer Science,https://www.scopus.com/record/display.url?eid=...,The development of modern technology has broug...,10.1109/ICVEE50212.2020.9243253,Conference Paper,NaN,Indonesia,Universitas Negeri Surabaya
26,Multiplane Convolutional Neural Network (Mp-CN...,"Angkoso, C.V.| Tjahyaningtijas, H.P.A.| Purnom...",57191037824| 58028094300| 6602604153| 35794232900,2022.0,International Journal of Intelligent Engineeri...,English,2.5,44,0.303,Computer Science| Engineering,https://www.scopus.com/record/display.url?eid=...,One of the objectives of medical imaging resea...,10.22266/IJIES2022.0228.30,Article,Bronze,Indonesia,Institut Teknologi Sepuluh Nopember| Universit...
27,Blind robust image watermarking based on adapt...,"Rakhmawati, L.| Wirawan, W.| Suwadi, S.| Delph...",57200562830| 7409815140| 35975312300| 66019276...,2022.0,Expert Systems with Applications,English,12.6,4,1.873,Engineering| Computer Science,https://www.scopus.com/record/display.url?eid=...,The image watermarking technique is a reliable...,10.1016/j.eswa.2021.115906,Article,NaN,Indonesia| France,Institut Teknologi Sepuluh Nopember| CNRS| Cen...
43,A Novel Improved Sea-Horse Optimizer for Tunin...,"Aribowo, W.",57216862548,2023.0,Journal of Robotics and Control (JRC),English,6.3,26,0.361,Computer Science| Engineering,https://www.scopus.com/record/display.url?eid=...,Power system stabilizer (PSS) is applied to da...,10.18196/jrc.v4i1.16445,Article,Gold,Indonesia,Universitas Negeri Surabaya


In [39]:
# === Filter Scival ke Computer Science + Enrich author Infokom via Scopus IDs (clean version) ===
import pandas as pd
import re
from pathlib import Path

# ------------ PATHS (silakan sesuaikan) ------------
SCIVAL_IN   = BASE / "file_tabulars/scival_scraped.csv"
DOSEN_IN    = BASE / "file_tabulars/dosen_data.csv"
SCIVAL_OUT  = BASE / "file_tabulars/scival_filtered_infokom.csv"

# ------------ Helpers ------------
def extract_scopus_ids_any(x):
    """Ambil semua Scopus Author ID (deretan angka) dari string apa pun (url/pipe/comma/spasi)."""
    if pd.isna(x):
        return []
    s = str(x).replace("\u00A0", " ").replace("\u200B", "")
    return re.findall(r"\d{7,13}", s)

def split_ids(s):
    """Pisah 'a| b' atau 'a, b' → list[str] (hanya angka), hilangkan kosong."""
    if pd.isna(s):
        return []
    s = str(s).replace("\u00A0", " ").replace("\u200B", "")
    parts = re.split(r"[|,;]|\s{2,}", s)
    out = []
    for p in parts:
        p = p.strip()
        if p and p.lower() != "nan":
            # normalisasi: hanya terima deretan angka (Scopus Author ID numerik)
            m = re.fullmatch(r"\d{7,13}", p)
            if m:
                out.append(p)
            else:
                out.extend(extract_scopus_ids_any(p))
    return list(dict.fromkeys(out))  # unik + pertahankan urutan

def pick_col_exact(df, candidates):
    low = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in low:
            return low[c.lower()]
    return None

def pick_col_contains(df, words):
    for c in df.columns:
        cl = c.lower()
        if all(w in cl for w in words):
            return c
    return None

# ------------ 1) Load & Filter THE: Computer Science ------------
df = pd.read_csv(SCIVAL_IN, encoding="utf-8-sig")

the_field_col = pick_col_contains(df, ["times higher education", "field name"])
assert the_field_col is not None, "Kolom THE field name tidak ditemukan."

# regex aman untuk nilai multi-label: "Computer Science| Education Studies| Engineering"
pat_cs = re.compile(r"(^|\|)\s*Computer Science\s*(\||$)", re.IGNORECASE)
df = df[df[the_field_col].astype(str).apply(lambda x: bool(pat_cs.search(x)))]

# ------------ 2) Simpan hanya kolom relevan ------------
keep_cols = [
    "title", "authors", "scopus author ids", "year",
    "scopus source title", "language",
    "citescore (publication year)", "citescore percentile (publication year) *",
    "sjr (publication year)",
    the_field_col,                 # THE field name
    "abstract_x", "abstract_y", "doi",
    "publication type", "open access",
    "country/region", "institutions",
]
keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols].copy()

# ------------ 3) Build lookup dosen: ScopusID → Nama Dosen ------------
dosen = pd.read_csv(DOSEN_IN, encoding="utf-8-sig")

dosen_name_col = pick_col_exact(dosen, ["nama_dosen","nama dosen","nama"]) or pick_col_contains(dosen, ["nama"])
assert dosen_name_col is not None, "Kolom nama dosen tidak ditemukan pada dosen_data.csv"

# kumpulkan semua kolom kandidat berisi Scopus Author ID
cand_scopus_cols = []
for c in dosen.columns:
    cl = c.lower()
    if ("scopus" in cl and "id" in cl) or ("author" in cl and "id" in cl and "scopus" in cl):
        cand_scopus_cols.append(c)
if not cand_scopus_cols:
    # fallback: kolom yang sering berisi angka panjang
    for c in dosen.columns:
        try:
            if dosen[c].astype(str).str.contains(r"\d{7,13}", regex=True, na=False).mean() > 0.2:
                cand_scopus_cols.append(c)
        except Exception:
            pass
assert cand_scopus_cols, "Tidak menemukan kolom Scopus Author ID di dosen_data.csv"

tmp = dosen[[dosen_name_col] + cand_scopus_cols].copy()

def unify_ids_row(row):
    bag = []
    for c in cand_scopus_cols:
        bag += extract_scopus_ids_any(row[c])
    return "|".join(sorted(set(bag)))

tmp["scopus_ids_unified"] = tmp.apply(unify_ids_row, axis=1)

dosen_expl = (
    tmp[[dosen_name_col, "scopus_ids_unified"]]
    .assign(scopus_ids_unified=lambda d: d["scopus_ids_unified"].fillna(""))
    .assign(_ids=lambda d: d["scopus_ids_unified"].apply(lambda s: [t for t in s.split("|") if t]))
    .explode("_ids")
    .rename(columns={dosen_name_col: "dosen_name", "_ids": "scopus_id"})
    .dropna(subset=["scopus_id"])
)
dosen_expl["scopus_id"] = dosen_expl["scopus_id"].astype(str).str.strip()
dosen_expl = dosen_expl[dosen_expl["scopus_id"] != ""]

id2names = (
    dosen_expl.groupby("scopus_id")["dosen_name"]
    .apply(lambda s: sorted(set(map(str, s))))
    .to_dict()
)

# ------------ 4) Tambahkan kolom author Infokom (by Scopus Author IDs) ------------
scival_authids_col = (
    pick_col_exact(df, ["scopus author ids","scopus author id","author ids","author id"]) or
    pick_col_contains(df, ["author","id"])
)
assert scival_authids_col is not None, "Kolom 'scopus author ids' tidak ditemukan di Scival."

infokom_names, infokom_ids = [], []
for ids_str in df[scival_authids_col].tolist():
    ids = split_ids(ids_str)
    hit_names, hit_ids = [], []
    for sid in ids:
        if sid in id2names:
            hit_ids.append(sid)
            hit_names.extend(id2names[sid])
    # dedup dengan menjaga urutan
    seen = set()
    hit_names = [x for x in hit_names if not (x in seen or seen.add(x))]
    seen = set()
    hit_ids = [x for x in hit_ids if not (x in seen or seen.add(x))]
    infokom_names.append("|".join(hit_names))
    infokom_ids.append("|".join(hit_ids))

df["infokom_author_names"]      = infokom_names
df["infokom_author_scopus_ids"] = infokom_ids
df["has_infokom_author"]        = df["infokom_author_scopus_ids"].astype(str).str.len().gt(0)

# ------------ 5) Simpan & ringkas ------------
df.to_csv(SCIVAL_OUT, index=False, encoding="utf-8-sig")

print("✅ Selesai.")
print("→ Tersimpan:", SCIVAL_OUT)
print("→ Baris tersisa (THE=Computer Science):", len(df))
print("→ Baris dengan minimal 1 author Infokom:", int(df["has_infokom_author"].sum()))


✅ Selesai.
→ Tersimpan: C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\scival_filtered_infokom.csv
→ Baris tersisa (THE=Computer Science): 453
→ Baris dengan minimal 1 author Infokom: 163
